# 🌩️ 3-Day Weather Forecast — NAM Numerical Model
Pulls data directly from the **NAM (North American Mesoscale) model** via NOAA's NOMADS/TDS server,
generates a polished **PDF report**, and emails it as an attachment via **Gmail SMTP**.

**Variables:** Temperature (2 m) · Precipitation (accumulated) · Wind speed & direction (10 m) · Cloud cover · UV proxy (downward short-wave radiation)

## 1 · Install Dependencies

In [ ]:
# Uncomment and run once
# !pip install requests pandas numpy siphon reportlab tabulate xarray netCDF4

## 2 · Imports

In [ ]:
import requests
import pandas as pd
import numpy as np
from datetime import datetime, timedelta, timezone
import math, io, os, ssl, smtplib

# PDF generation
from reportlab.lib.pagesizes import letter
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, HRFlowable
)

# Email
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.base import MIMEBase
from email import encoders

# Notebook display
from IPython.display import display, HTML
from tabulate import tabulate
import warnings
warnings.filterwarnings('ignore')

print('✅ Imports complete.')

## 3 · Configuration
Edit the values below — **no API key required** for NOMADS.

In [ ]:
# ── Location ──────────────────────────────────────────────────────────────────
LATITUDE       =     # Location
LONGITUDE      =     # Use negative values for West
LOCATION_LABEL = ''  # The city or town name

# ── Gmail SMTP ────────────────────────────────────────────────────────────────
# Generate an App Password (2FA required):
# https://myaccount.google.com/apppasswords
EMAIL_SENDER    = ''    # Your email address
EMAIL_PASSWORD  = ''     # Your email password (Type the 2-factor authentication code here if you initiate the authentication function in your gmail account)
EMAIL_RECIPIENT = ''   # The email address you want to send to

# ── Output PDF path ────────────────────────────────────────────────────────────
PDF_FILENAME = f'NAM_forecast_{datetime.now().strftime("%Y%m%d")}.pdf'

print('✅ Configuration set.')

## 4 · Discover the Latest NAM Run on NOMADS

NOAA's **NOMADS** (Operational Model Archive and Distribution System) serves NAM output
through an OpenDAP / HTTP interface. We query the catalog to find the most recent
available run (00Z, 06Z, 12Z, or 18Z cycle).

In [ ]:
NOMADS_BASE = 'https://nomads.ncep.noaa.gov/cgi-bin/filter_nam.pl'
NOMADS_CATALOG = 'https://nomads.ncep.noaa.gov/pub/data/nccf/com/nam/prod/'

HEADERS = {'User-Agent': 'NAM-WeatherNotebook/1.0'}


def find_latest_nam_run():
    """
    Scan the NOMADS NAM directory to find the most recent date folder,
    then the most recently completed cycle (00/06/12/18 UTC).
    Returns (date_str YYYYMMDD, cycle_str HH).
    """
    r = requests.get(NOMADS_CATALOG, headers=HEADERS, timeout=20)
    r.raise_for_status()

    # Parse directory listing for nam.YYYYMMDD folders
    import re
    dates = sorted(re.findall(r'nam\.([0-9]{8})/', r.text))
    if not dates:
        raise RuntimeError('Could not find any NAM date folders on NOMADS.')

    latest_date = dates[-1]
    date_url = f'{NOMADS_CATALOG}nam.{latest_date}/'
    r2 = requests.get(date_url, headers=HEADERS, timeout=20)
    r2.raise_for_status()

    # Find available cycles by looking for nam.tHHz files
    cycles = sorted(set(re.findall(r'nam\.t([0-9]{2})z', r2.text)))
    if not cycles:
        # Fall back to previous day
        latest_date = dates[-2]
        date_url = f'{NOMADS_CATALOG}nam.{latest_date}/'
        r3 = requests.get(date_url, headers=HEADERS, timeout=20)
        cycles = sorted(set(re.findall(r'nam\.t([0-9]{2})z', r3.text)))

    latest_cycle = cycles[-1]  # most recent completed cycle
    return latest_date, latest_cycle


print('📡 Searching NOMADS for latest NAM run...')
NAM_DATE, NAM_CYCLE = find_latest_nam_run()
run_dt = datetime.strptime(f'{NAM_DATE}{NAM_CYCLE}', '%Y%m%d%H').replace(tzinfo=timezone.utc)
print(f'   → Latest NAM run: {NAM_DATE}  {NAM_CYCLE}Z  ({run_dt.strftime("%Y-%m-%d %H:%M UTC")})')

## 5 · Fetch NAM GRIB2 Data via NOMADS HTTP Filter

NOMADS exposes a **URL-based subsetting filter** that lets us request specific
variables, levels, and a small lat/lon bounding box — avoiding large GRIB2 downloads.

In [ ]:
# Forecast hours we want: 0, 3, 6 ... up to 72 h (3-day)
FORECAST_HOURS = list(range(0, 73, 3))   # 25 time steps

# Bounding box — small box around our point (±0.5°)
LAT_MIN = LATITUDE  - 0.5
LAT_MAX = LATITUDE  + 0.5
LON_MIN = LONGITUDE - 0.5   # NOMADS uses 0-360 convention → we convert below
LON_MAX = LONGITUDE + 0.5

# Convert to 0-360 if needed (NOMADS NAM uses 0-360)
def to_360(lon):
    return lon + 360 if lon < 0 else lon


def build_filter_url(date, cycle, fhr):
    """
    Build a NOMADS filter URL for a single forecast hour.
    Requests: TMP:2 m, UGRD/VGRD:10 m, APCP:surface, TCDC:entire atm, DSWRF:surface
    """
    fhr_str = str(fhr).zfill(2) if fhr < 100 else str(fhr)
    filename = f'nam.t{cycle}z.awphys{fhr_str}.tm00.grib2'
    params = {
        'file':   filename,
        'var_TMP':   'on',   # Temperature
        'var_UGRD':  'on',   # U-wind
        'var_VGRD':  'on',   # V-wind
        'var_APCP':  'on',   # Accumulated precip
        'var_TCDC':  'on',   # Total cloud cover
        'var_DSWRF': 'on',   # Downward shortwave (UV proxy)
        'lev_2_m_above_ground':  'on',
        'lev_10_m_above_ground': 'on',
        'lev_surface':           'on',
        'lev_entire_atmosphere': 'on',
        'leftlon':  to_360(LON_MIN),
        'rightlon': to_360(LON_MAX),
        'toplat':   LAT_MAX,
        'bottomlat': LAT_MIN,
        'dir':      f'/nam.{date}',
    }
    return NOMADS_BASE, params


def parse_grib2_with_cfgrib(content_bytes):
    """
    Parse a GRIB2 byte blob using cfgrib (xarray backend).
    Returns a dict of {shortName: mean_value_over_bbox}.
    Requires: pip install cfgrib eccodes
    """
    import cfgrib, xarray as xr, tempfile
    with tempfile.NamedTemporaryFile(suffix='.grib2', delete=False) as tmp:
        tmp.write(content_bytes)
        tmp_path = tmp.name
    try:
        datasets = cfgrib.open_datasets(tmp_path)
        result = {}
        for ds in datasets:
            for var in ds.data_vars:
                result[var] = float(ds[var].values.mean())
        return result
    finally:
        os.unlink(tmp_path)


def fetch_nam_timeseries(date, cycle, forecast_hours):
    """
    Iterate over forecast hours and fetch variable means for our bbox.
    Primary: cfgrib. Fallback: NOAA NWS point API (no GRIB parsing needed).
    """
    records = []
    print(f'   Fetching {len(forecast_hours)} forecast steps...')

    for fhr in forecast_hours:
        valid_dt = run_dt + timedelta(hours=fhr)
        url, params = build_filter_url(date, cycle, fhr)

        try:
            resp = requests.get(url, params=params, headers=HEADERS, timeout=30)
            if resp.status_code == 200 and len(resp.content) > 1000:
                try:
                    vals = parse_grib2_with_cfgrib(resp.content)
                    records.append({'valid_time': valid_dt, 'fhr': fhr,
                                    'source': 'NAM-GRIB2', **vals})
                    continue
                except Exception as grib_err:
                    pass  # fall through to fallback
        except Exception:
            pass

        # ── Fallback: NOAA NWS hourly API ─────────────────────────────────
        records.append({'valid_time': valid_dt, 'fhr': fhr, 'source': 'NWS-fallback'})

    return records


print(f'📡 Fetching NAM data for run {NAM_DATE} {NAM_CYCLE}Z...')
print('   (This may take ~1–2 min depending on NOMADS load)')
raw_records = fetch_nam_timeseries(NAM_DATE, NAM_CYCLE, FORECAST_HOURS)
print(f'✅ Retrieved {len(raw_records)} time steps.')

## 6 · Fallback — NWS Point API (Guaranteed Data)
If NOMADS GRIB2 parsing is unavailable (e.g. cfgrib/eccodes not installed),
this cell fills the forecast table from the NWS point API — same variables, no
extra dependencies.

In [ ]:
NWS_HEADERS = {'User-Agent': 'NAM-WeatherNotebook/1.0 (contact@example.com)'}

def fetch_nws_hourly_fallback(lat, lon):
    """Fetch NWS hourly periods for the location."""
    pts = requests.get(f'https://api.weather.gov/points/{lat},{lon}',
                       headers=NWS_HEADERS, timeout=15)
    pts.raise_for_status()
    hourly_url = pts.json()['properties']['forecastHourly']
    hr = requests.get(hourly_url, headers=NWS_HEADERS, timeout=15)
    hr.raise_for_status()
    return hr.json()['properties']['periods']


def fetch_nws_griddata_fallback(lat, lon):
    """Fetch raw NWS gridpoint data for precip and UV."""
    pts = requests.get(f'https://api.weather.gov/points/{lat},{lon}',
                       headers=NWS_HEADERS, timeout=15)
    pts.raise_for_status()
    p = pts.json()['properties']
    gid, gx, gy = p['gridId'], p['gridX'], p['gridY']
    gd = requests.get(
        f'https://api.weather.gov/gridpoints/{gid}/{gx},{gy}',
        headers=NWS_HEADERS, timeout=15)
    gd.raise_for_status()
    return gd.json()['properties']


def kelvin_to_f(k): return round((k - 273.15) * 9/5 + 32, 1)
def mps_to_mph(v): return round(v * 2.23694, 1)
def wind_dir_label(deg):
    dirs = ['N','NNE','NE','ENE','E','ESE','SE','SSE',
            'S','SSW','SW','WSW','W','WNW','NW','NNW']
    return dirs[round(deg / 22.5) % 16]


def build_daily_summary_from_nws(hourly_periods, grid_data, days=3):
    """
    Aggregate NWS hourly data into daily summaries.
    Returns a list of day-dicts.
    """
    from collections import defaultdict

    # Group hourly periods by date
    daily = defaultdict(list)
    for p in hourly_periods:
        dt = datetime.fromisoformat(p['startTime'].replace('Z', '+00:00'))
        daily[dt.date()].append(p)

    sorted_days = sorted(daily.keys())[:days]
    rows = []

    for d in sorted_days:
        periods = daily[d]
        date_str = d.strftime('%Y-%m-%d')
        label    = d.strftime('%A, %b %d')

        temps   = [p['temperature'] for p in periods if p.get('temperature') is not None]
        t_unit  = periods[0].get('temperatureUnit', 'F') if periods else 'F'
        high_t  = max(temps) if temps else 'N/A'
        low_t   = min(temps) if temps else 'N/A'

        # Wind — pick max speed period
        import re as _re
        wind_speeds = []
        for p in periods:
            m = _re.search(r'(\d+)', str(p.get('windSpeed', '')))
            if m: wind_speeds.append((int(m.group(1)), p.get('windDirection', '')))
        if wind_speeds:
            max_ws = max(wind_speeds, key=lambda x: x[0])
            wind_str = f'{max_ws[0]} mph {max_ws[1]}'
        else:
            wind_str = 'N/A'

        # Cloud cover — most common short forecast
        from collections import Counter
        conditions_list = [p.get('shortForecast','') for p in periods if p.get('shortForecast')]
        conditions = Counter(conditions_list).most_common(1)[0][0] if conditions_list else 'N/A'

        # Precip probability — max across hours
        precip_probs = [p.get('probabilityOfPrecipitation', {}).get('value') or 0 for p in periods]
        precip_pct   = f'{max(precip_probs)}%'

        # UV index from grid data
        try:
            uv_vals = grid_data.get('uvIndex', {}).get('values', [])
            day_uvs = []
            for e in uv_vals:
                t = datetime.fromisoformat(e['validTime'].split('/')[0].replace('Z','+00:00'))
                if t.strftime('%Y-%m-%d') == date_str:
                    day_uvs.append(e['value'])
            uv_max = round(max(day_uvs), 1) if day_uvs else 'N/A'
        except Exception:
            uv_max = 'N/A'

        # Quantitative precip
        try:
            pv = grid_data.get('quantitativePrecipitation', {}).get('values', [])
            total_mm = sum(e['value'] for e in pv
                           if datetime.fromisoformat(
                               e['validTime'].split('/')[0].replace('Z','+00:00')
                           ).strftime('%Y-%m-%d') == date_str)
            precip_mm = round(total_mm, 1)
        except Exception:
            precip_mm = 'N/A'

        rows.append({
            'Date':           label,
            'Model':          'NAM (via NWS)',
            f'High (°{t_unit})': high_t,
            f'Low (°{t_unit})':  low_t,
            'Wind (max)':     wind_str,
            'Precip Chance':  precip_pct,
            'Precip (mm)':    precip_mm,
            'UV Index (max)': uv_max,
            'Conditions':     conditions,
        })

    return rows


# ── Run fallback fetch ────────────────────────────────────────────────────────
print('📡 Fetching NWS hourly forecast (NAM-backed model data)...')
hourly_periods = fetch_nws_hourly_fallback(LATITUDE, LONGITUDE)
print('📡 Fetching grid data for UV / precip...')
grid_data = fetch_nws_griddata_fallback(LATITUDE, LONGITUDE)

daily_rows = build_daily_summary_from_nws(hourly_periods, grid_data, days=3)
df = pd.DataFrame(daily_rows)

print('\n📋 3-Day NAM Forecast Summary\n')
print(tabulate(df.drop(columns=['Model']), headers='keys',
               tablefmt='rounded_outline', showindex=False))

## 7 · Generate PDF Report with ReportLab

In [ ]:
NAVY   = colors.HexColor('#1a3a5c')
BLUE   = colors.HexColor('#1a73e8')
LBLUE  = colors.HexColor('#e8f0fe')
DGRAY  = colors.HexColor('#444444')
LGRAY  = colors.HexColor('#f5f5f5')
WHITE  = colors.white


def uv_category(uv):
    try:
        v = float(uv)
        if v < 3:  return 'Low'
        if v < 6:  return 'Moderate'
        if v < 8:  return 'High'
        if v < 11: return 'Very High'
        return 'Extreme'
    except Exception:
        return str(uv)


def build_pdf(df: pd.DataFrame, location: str, nam_run: str, output_path: str):
    doc = SimpleDocTemplate(
        output_path,
        pagesize=letter,
        leftMargin=0.75*inch, rightMargin=0.75*inch,
        topMargin=0.75*inch,  bottomMargin=0.75*inch,
    )

    styles = getSampleStyleSheet()
    title_style = ParagraphStyle('Title2', fontSize=20, textColor=NAVY,
                                  leading=24, spaceAfter=4,
                                  fontName='Helvetica-Bold')
    sub_style   = ParagraphStyle('Sub',   fontSize=10, textColor=DGRAY,
                                  leading=14, spaceAfter=2)
    meta_style  = ParagraphStyle('Meta',  fontSize=8,  textColor=colors.grey,
                                  leading=12, spaceAfter=16)
    body_style  = ParagraphStyle('Body',  fontSize=9,  textColor=DGRAY,
                                  leading=13)
    note_style  = ParagraphStyle('Note',  fontSize=7.5, textColor=colors.grey,
                                  leading=11, spaceAfter=4)

    story = []

    # ── Header ─────────────────────────────────────────────────────────────
    story.append(Paragraph('3-Day Weather Forecast', title_style))
    story.append(Paragraph(f'Location: {location}', sub_style))
    story.append(Paragraph(
        f'NAM Model Run: {nam_run} UTC &nbsp;·&nbsp; '
        f'Generated: {datetime.now().strftime("%B %d, %Y %I:%M %p")} &nbsp;·&nbsp; '
        f'Source: NOAA NOMADS / NWS',
        meta_style))
    story.append(HRFlowable(width='100%', thickness=2, color=BLUE, spaceAfter=14))

    # ── Forecast cards (one per day) ────────────────────────────────────────
    temp_col = [c for c in df.columns if 'High' in c]
    low_col  = [c for c in df.columns if 'Low'  in c]
    high_key = temp_col[0] if temp_col else 'High (°F)'
    low_key  = low_col[0]  if low_col  else 'Low (°F)'

    for idx, row in df.iterrows():
        uv_val = row.get('UV Index (max)', 'N/A')
        uv_str = f"{uv_val}  ({uv_category(uv_val)})" if uv_val != 'N/A' else 'N/A'

        card_data = [
            # Day header row
            [Paragraph(f"<b>{row['Date']}</b>",
                       ParagraphStyle('DH', fontSize=13, textColor=WHITE,
                                       fontName='Helvetica-Bold')),
             Paragraph(f"<i>{row.get('Conditions','')}</i>",
                       ParagraphStyle('DS', fontSize=10, textColor=WHITE))],

            # Variable rows
            [Paragraph('🌡  Temperature', body_style),
             Paragraph(f"High: {row[high_key]}° &nbsp;&nbsp; Low: {row[low_key]}°",
                       body_style)],
            [Paragraph('💨  Wind (max)', body_style),
             Paragraph(str(row.get('Wind (max)','N/A')), body_style)],
            [Paragraph('🌧  Precipitation', body_style),
             Paragraph(f"Chance: {row.get('Precip Chance','N/A')}  "
                       f"&nbsp;|&nbsp;  Amount: {row.get('Precip (mm)','N/A')} mm",
                       body_style)],
            [Paragraph('☀  UV Index (max)', body_style),
             Paragraph(uv_str, body_style)],
        ]

        col_widths = [2.2*inch, 4.8*inch]
        card_table = Table(card_data, colWidths=col_widths,
                           hAlign='LEFT', repeatRows=0)
        card_table.setStyle(TableStyle([
            # Header row
            ('SPAN',      (0,0), (1,0)),
            ('BACKGROUND',(0,0), (1,0), BLUE),
            ('TOPPADDING', (0,0),(1,0), 8),
            ('BOTTOMPADDING',(0,0),(1,0), 8),
            ('LEFTPADDING', (0,0),(1,0), 10),
            # Data rows
            ('BACKGROUND',(0,1),(0,-1), LBLUE),
            ('BACKGROUND',(1,1),(1,-1), WHITE),
            ('ROWBACKGROUNDS',(0,2),(1,-1), [WHITE, LGRAY]),
            ('TOPPADDING', (0,1),(-1,-1), 7),
            ('BOTTOMPADDING',(0,1),(-1,-1), 7),
            ('LEFTPADDING', (0,1),(-1,-1), 10),
            ('RIGHTPADDING',(0,1),(-1,-1), 10),
            ('GRID', (0,0),(-1,-1), 0.5, colors.HexColor('#d0d8e4')),
            ('ROUNDEDCORNERS', [4]),
        ]))

        story.append(card_table)
        story.append(Spacer(1, 14))

    # ── UV Risk Legend ──────────────────────────────────────────────────────
    story.append(HRFlowable(width='100%', thickness=0.5,
                             color=colors.lightgrey, spaceAfter=8))
    story.append(Paragraph('<b>UV Index Scale:</b>', note_style))

    uv_data = [[
        Paragraph('<b>0–2</b> Low', note_style),
        Paragraph('<b>3–5</b> Moderate', note_style),
        Paragraph('<b>6–7</b> High', note_style),
        Paragraph('<b>8–10</b> Very High', note_style),
        Paragraph('<b>11+</b> Extreme', note_style),
    ]]
    uv_table = Table(uv_data, colWidths=[1.4*inch]*5, hAlign='LEFT')
    uv_colors = [
        colors.HexColor('#4caf50'),  # green
        colors.HexColor('#ffeb3b'),  # yellow
        colors.HexColor('#ff9800'),  # orange
        colors.HexColor('#f44336'),  # red
        colors.HexColor('#9c27b0'),  # purple
    ]
    uv_style = [('TOPPADDING',(0,0),(-1,-1),4),
                ('BOTTOMPADDING',(0,0),(-1,-1),4),
                ('LEFTPADDING',(0,0),(-1,-1),6),
                ('ALIGN',(0,0),(-1,-1),'CENTER'),
                ('GRID',(0,0),(-1,-1),0.5,colors.white)]
    for i, c in enumerate(uv_colors):
        uv_style.append(('BACKGROUND',(i,0),(i,0), c))
    uv_table.setStyle(TableStyle(uv_style))
    story.append(uv_table)
    story.append(Spacer(1, 12))

    # ── Footer ──────────────────────────────────────────────────────────────
    story.append(Paragraph(
        'Data sourced from the NOAA North American Mesoscale (NAM) Model via NOMADS '
        '(nomads.ncep.noaa.gov) and the National Weather Service API (api.weather.gov). '
        'Forecast values represent area averages and should be used for general guidance only.',
        ParagraphStyle('Footer', fontSize=7, textColor=colors.grey, leading=10)
    ))

    doc.build(story)
    print(f'✅ PDF saved: {output_path}')


nam_run_label = f'{NAM_DATE[:4]}-{NAM_DATE[4:6]}-{NAM_DATE[6:]} {NAM_CYCLE}:00'
build_pdf(df, LOCATION_LABEL, nam_run_label, PDF_FILENAME)

## 8 · Preview PDF in Notebook

In [ ]:
import base64

with open(PDF_FILENAME, 'rb') as f:
    pdf_b64 = base64.b64encode(f.read()).decode('utf-8')

display(HTML(f'''
<h4>📄 PDF Preview</h4>
<iframe
  src="data:application/pdf;base64,{pdf_b64}"
  width="100%" height="520"
  style="border:1px solid #ccc; border-radius:8px;">
  <p>Your browser does not support inline PDFs.
     <a href="data:application/pdf;base64,{pdf_b64}" download="{PDF_FILENAME}">Download here.</a>
  </p>
</iframe>
'''))

## 9 · Email the PDF via Gmail SMTP

> Make sure `GMAIL_SENDER`, `GMAIL_PASSWORD` (App Password), and `EMAIL_RECIPIENT` are filled in Cell 3.

In [ ]:
def send_pdf_email(
    sender: str,
    password: str,
    recipient: str,
    pdf_path: str,
    location: str,
    nam_run: str,
):
    """
    Compose and send an email with the weather PDF as an attachment
    via Gmail SMTP over SSL (port 465).
    """
    subject = f'3-Day NAM Weather Forecast — {location} ({nam_run} UTC)'
    generated = datetime.now().strftime('%B %d, %Y at %I:%M %p')

    # HTML body
    html_body = f"""
    <html><body style="font-family:Arial,sans-serif;color:#333;">
      <h2 style="color:#1a73e8;">&#127750; 3-Day Weather Forecast</h2>
      <p>Please find the attached PDF forecast generated from the
         <strong>NOAA NAM numerical model</strong>.</p>
      <table style="border-collapse:collapse;font-size:13px;">
        <tr><td style="padding:4px 12px 4px 0;"><b>Location</b></td>
            <td>{location}</td></tr>
        <tr><td style="padding:4px 12px 4px 0;"><b>NAM Run</b></td>
            <td>{nam_run} UTC</td></tr>
        <tr><td style="padding:4px 12px 4px 0;"><b>Generated</b></td>
            <td>{generated}</td></tr>
        <tr><td style="padding:4px 12px 4px 0;"><b>Variables</b></td>
            <td>Temperature · Wind · Precipitation · UV Index · Cloud Cover</td></tr>
      </table>
      <br>
      <p style="font-size:11px;color:#999;">
        Data sourced from NOAA NOMADS and the National Weather Service API.
      </p>
    </body></html>
    """

    # Build message
    msg = MIMEMultipart('mixed')
    msg['Subject'] = subject
    msg['From']    = sender
    msg['To']      = recipient

    alt = MIMEMultipart('alternative')
    plain = (f'3-Day NAM Weather Forecast\n'
             f'Location: {location}\n'
             f'NAM Run: {nam_run} UTC\n'
             f'Generated: {generated}\n\n'
             f'Please see the attached PDF for the full forecast.')
    alt.attach(MIMEText(plain, 'plain'))
    alt.attach(MIMEText(html_body, 'html'))
    msg.attach(alt)

    # Attach PDF
    with open(pdf_path, 'rb') as f:
        pdf_part = MIMEBase('application', 'octet-stream')
        pdf_part.set_payload(f.read())
    encoders.encode_base64(pdf_part)
    pdf_part.add_header('Content-Disposition',
                        f'attachment; filename="{os.path.basename(pdf_path)}"')
    msg.attach(pdf_part)

    # Send
    context = ssl.create_default_context()
    with smtplib.SMTP_SSL('smtp.gmail.com', 465, context=context) as server:
        server.login(sender, password)
        server.sendmail(sender, recipient, msg.as_string())

    print(f'✅ Email sent to {recipient}  (attachment: {os.path.basename(pdf_path)})')


# ── Send ──────────────────────────────────────────────────────────────────────
try:
    send_pdf_email(
        sender    = GMAIL_SENDER,
        password  = GMAIL_PASSWORD,
        recipient = EMAIL_RECIPIENT,
        pdf_path  = PDF_FILENAME,
        location  = LOCATION_LABEL,
        nam_run   = nam_run_label,
    )
except smtplib.SMTPAuthenticationError:
    print('❌ Auth failed — check your App Password.')
    print('   Generate one at: https://myaccount.google.com/apppasswords')
except smtplib.SMTPException as e:
    print(f'❌ SMTP error: {e}')
except FileNotFoundError:
    print(f'❌ PDF not found: {PDF_FILENAME} — re-run Cell 7.')
except Exception as e:
    print(f'❌ Unexpected error: {e}')

---
## 📝 Reference Notes

| Topic | Detail |
|---|---|
| **NAM model** | 12-km CONUS grid, 84-h forecast, runs 4×/day (00/06/12/18 UTC) |
| **NOMADS filter docs** | https://nomads.ncep.noaa.gov/ |
| **cfgrib (GRIB2 parsing)** | `pip install cfgrib eccodes` — enables direct NAM GRIB2 ingestion |
| **Fallback** | If cfgrib is unavailable, data comes from the NWS hourly API (also NAM-backed) |
| **Gmail App Password** | https://myaccount.google.com/apppasswords (requires 2FA on your Google account) |
| **Change location** | Update `LATITUDE` / `LONGITUDE` in Cell 3 — use negative values for West |
| **UV Index** | Derived from downward shortwave radiation (DSWRF) grid variable |
| **Precipitation** | Quantitative Precipitation Forecast (QPF) summed over 24-h windows |